In [1]:
 
#!pip install autogen-agentchat==0.4.7"
#!pip install -U "autogen-ext[openai]==0.4.7" chromadb
#!pip uninstall agentops opentelemetry=instrumentation opentelemetry-sdk -y

In [2]:
#!pip install openai-agents

In [3]:
import asyncio
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_core.tools import FunctionTool
import chromadb

# 1. Use a NEW key - revoke the one you leaked 5 times
os.environ["OPENAI_API_KEY"] = "put api key"

# 2. OpenAI client for 0.4+ - replaces all llm_config/config_list stuff
client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [4]:
#!pip install python-docx

In [5]:
import asyncio
import os
import chromadb
from docx import Document
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_core.tools import FunctionTool

os.environ["OPENAI_API_KEY"] = "put api key"

client = OpenAIChatCompletionClient(model="gpt-4o-mini")


# Paste your real path here
docx_path = "C:/Users/KAVITHA/Chola.docx"
doc = Document(docx_path)
kb_text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])                   
print("Does file exist?", os.path.exists(docx_path))

print("First 200 chars:",kb_text[:200])
import os

# Check where your notebook is
print("Notebook folder:", os.getcwd())

# List files in your notebook folder
print("\nFiles here:")
for f in os.listdir():
    print(" -", f)

# Check the 2 most likely folders
folder1 = "C:/Users/KAVITHA/AutoGen/1.Conversationalchat-Rag"  # Guess the full name
folder2 = "C:/Users/KAVITHA/AutoGen/2.Two Agents-Retrivalchat-Rag"

print("\nChecking folder1:", os.path.exists(folder1))
if os.path.exists(folder1):
    print("Files in folder1:", os.listdir(folder1))

print("\nChecking folder2:", os.path.exists(folder2)) 
if os.path.exists(folder2):
    print("Files in folder2:", os.listdir(folder2))

# Read your.docx file correctly
def read_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs if p.text])

kb_text = read_docx("C:/Users/KAVITHA/Chola.docx")

# Load into ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("temple_docs")
collection.add(documents=[kb_text], ids=["temple_1"])

def search_docs(query: str) -> str:
    results = collection.query(query_texts=[query], n_results=2)
    return "\n".join(results['documents'][0]) if results['documents'][0] else "No docs found"

search_tool = FunctionTool(search_docs, description="Search the temple document")

rag_agent = AssistantAgent(
    name="rag_agent",
    model_client=client,
    tools=[search_tool],
    system_message="Call search_docs before answering questions about the temple. Reply TERMINATE when done."
)

writer_agent = AssistantAgent(
    name="writer",
    model_client=client,
    system_message="Summarize the findings clearly. Reply TERMINATE when done."
)

team = RoundRobinGroupChat(
    [rag_agent, writer_agent],
    termination_condition=TextMentionTermination("TERMINATE")
)

await team.run(task="Summarize the key points about the temple")

Does file exist? True
First 200 chars: The Chola Dynasty was a prominent Tamil empire in Southern India that flourished between the 9th and 13th centuries. Renowned for its maritime power, advanced village self-government, and magnificent 
Notebook folder: C:\Users\KAVITHA\AutoGen\1.Converseable Agent-20260612T095424Z-3-001\1.Converseable Agent\2.Two Agents-Retrivalchat-Rag

Files here:
 - .cache
 - .ipynb_checkpoints
 - 1)RetrivalBasedTwoAgents_openai.ipynb
 - 2)RetrivalBasedTwoAgents_Gemini.ipynb
 - 3)RetrivalBasedTwoAgents_Ollama.ipynb
 - ABOUT _THE _TEMPLE.docx
 - ABOUT_THE_TEMPLE.pdf
 - agentops.log
 - anaconda_projects
 - autogen-docs
 - book.pdf
 - OAG_CONFIG_LIST.json
 - retrieve based two agents open ai 2.ipynb
 - retrieve based two agents openai 1.ipynb
 - Retrive two agents New openai.ipynb
 - tmp
 - Two agents rag openai.ipynb

Checking folder1: False

Checking folder2: False


TaskResult(messages=[TextMessage(source='user', models_usage=None, content='Summarize the key points about the temple', type='TextMessage'), ToolCallRequestEvent(source='rag_agent', models_usage=RequestUsage(prompt_tokens=74, completion_tokens=15), content=[FunctionCall(id='call_3gk3XwL5yeksJkEjTZ1fn5WK', arguments='{"query":"temple"}', name='search_docs')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='rag_agent', models_usage=None, content=[FunctionExecutionResult(content='The Chola Dynasty was a prominent Tamil empire in Southern India that flourished between the 9th and 13th centuries. Renowned for its maritime power, advanced village self-government, and magnificent Dravidian architecture, the empire dominated much of South India, Sri Lanka, and Southeast Asia at its peak. \nOrigins and Rise\nAncient Roots: Early Chola rulers are mentioned in Sangam literature and Ashokan edicts dating back to the 3rd century BCE.\nThe Imperial Revival: The dynasty was revived in th